# Evento Evaluativo 4 · Machine Learning · Intención de compra en e-commerce
## Notebook 01 — Descripción del problema y análisis exploratorio (EDA)

**Grupo 3 · Análisis de Datos · ITM 2026-2 · Docente: Daniel Alexis Nieto Mora**
Integrantes: Santiago Betancur (@Xantyagoh) · Diego Alzate (@DiegoA1123) · Carlos Mendez (@carmarisx) · Sebastian Cadavid (@Exmen9) · Daniel Martinez (@danielmmnez)

**Qué hace este notebook:** describe el problema, presenta el dataset, audita su calidad y lo explora (univariado y bivariado) para decidir el preprocesamiento (notebook 02) y los modelos (notebooks 03 y 04).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas del proyecto (los notebooks viven en evaluacion 4/notebooks)
RAIZ = Path('..').resolve()
DATOS = RAIZ / 'data' / 'online_shoppers_intention.csv'
FIG = RAIZ / 'figuras'
RES = RAIZ / 'resultados'
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

# Estilo comun a todos los graficos del proyecto
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, NARANJA, GRIS, VERDE, ROJO = '#1F4E79', '#E8833A', '#8C8C8C', '#3B8E5A', '#C0392B'
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Descripción del problema

Un comercio electrónico registra miles de **sesiones de navegación**: cuántas páginas vio cada visitante, cuánto tiempo estuvo, desde dónde llegó y si al final **compró o no**. La mayoría de las sesiones **no termina en compra**, así que el negocio necesita dos cosas distintas:

| Componente | Pregunta de negocio | Tipo de aprendizaje |
|---|---|---|
| **Supervisado** | Dada una sesión, ¿**terminará en compra**? (variable objetivo `Revenue`) | Clasificación binaria con desbalance |
| **No supervisado** | ¿Qué **tipos de visitantes** hay según su forma de navegar? (sin mirar `Revenue`) | Agrupamiento (segmentación) |

**Por qué importa:** predecir la intención de compra permite actuar mientras el visitante aún navega (un cupón, un chat de ayuda); segmentar permite diseñar estrategias distintas para visitantes exploratorios, comprometidos o que rebotan.

### Preguntas que queremos responder
1. ¿Qué tan desbalanceada está la compra y cómo afecta la elección de métricas?
2. ¿Qué características de la sesión se relacionan con comprar?
3. ¿Qué tan bien se puede predecir la compra y **cuánto depende de una sola variable** (`PageValues`)?
4. ¿Existen segmentos de visitantes con comportamientos claramente distintos?
5. ¿Los segmentos descubiertos **sin etiquetas** coinciden con quienes compran?
6. ¿Qué enfoque recomendaríamos para una aplicación real?

## 2. El dataset

**Dataset:** *Online Shoppers Purchasing Intention Dataset* — UCI Machine Learning Repository (id 468).
**Cita:** Sakar, C. & Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5F88Q · Licencia CC BY 4.0.
**Artículo asociado:** Sakar, C. O., Polat, S. O., Katircioglu, M., & Kastro, Y. (2019). Real-time prediction of online shoppers' purchasing intention using multilayer perceptron and LSTM recurrent neural networks. *Neural Computing and Applications*, 31, 6893–6908.
**Contenido:** 12.330 sesiones, 17 variables + `Revenue`. Cada sesión pertenece a **un usuario distinto** dentro de un periodo de un año (para evitar sesgos por campaña, fecha especial o perfil de usuario). Sin valores faltantes.

In [2]:
df = pd.read_csv(DATOS)
print('Sesiones:', f'{df.shape[0]:,}', '| Variables:', df.shape[1])
df.head()

Sesiones: 12,330 | Variables: 18


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.000,0,0.000,2,64.000,0.000,0.100,0.000,0.000,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.000,0,0.000,1,0.000,0.200,0.200,0.000,0.000,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.000,0,0.000,2,2.667,0.050,0.140,0.000,0.000,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.000,0,0.000,10,627.500,0.020,0.050,0.000,0.000,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

### Diccionario de datos

| Variable | Tipo | Significado |
|---|---|---|
| `Administrative`, `Administrative_Duration` | numérica | Nº de páginas administrativas (cuenta, perfil) visitadas y tiempo total en ellas (segundos) |
| `Informational`, `Informational_Duration` | numérica | Ídem para páginas informativas (contacto, ayuda, la empresa) |
| `ProductRelated`, `ProductRelated_Duration` | numérica | Ídem para páginas de **productos** |
| `BounceRates` | numérica (0–1) | % de visitas que entran a la página y se van sin hacer nada más (Google Analytics) |
| `ExitRates` | numérica (0–1) | % de vistas de la página que fueron la **última** de la sesión (Google Analytics) |
| `PageValues` | numérica | Valor promedio de las páginas visitadas **antes de completar una transacción** (Google Analytics) |
| `SpecialDay` | numérica (0–1) | Cercanía de la visita a una fecha especial (San Valentín, Día de la Madre…); 1 = máxima cercanía |
| `Month` | categórica | Mes de la visita (10 meses; no hay enero ni abril) |
| `OperatingSystems`, `Browser`, `Region`, `TrafficType` | categórica (códigos) | Sistema operativo, navegador, región y tipo de tráfico, codificados como números **sin orden** |
| `VisitorType` | categórica | `Returning_Visitor`, `New_Visitor` u `Other` |
| `Weekend` | binaria | ¿La visita fue en fin de semana? |
| **`Revenue`** | **binaria (objetivo)** | **¿La sesión terminó en compra?** |

**Unidad de observación:** una sesión de navegación.

In [4]:
ficha = pd.Series({
    'Sesiones': f'{len(df):,}',
    'Variables predictoras': df.shape[1] - 1,
    'Numericas': 10, 'Categoricas (codigos o texto)': 6, 'Binarias': 1,
    'Sesiones que terminan en compra': f'{df["Revenue"].sum():,} ({df["Revenue"].mean():.1%})',
    'Meses presentes': df['Month'].nunique(),
}, name='valor').to_frame()
ficha

,valor
Sesiones,"12,330"
Variables predictoras,17
Numericas,10
Categoricas (codigos o texto),6
Binarias,1
Sesiones que terminan en compra,"1,908 (15.5%)"
Meses presentes,10


## 3. Calidad de los datos

Antes de explorar se revisa si hay faltantes, duplicados, valores imposibles y variables problemáticas. **Cada hallazgo condiciona una decisión de preprocesamiento.**

In [5]:
calidad = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'valores_unicos': df.nunique(),
    'min': df.select_dtypes('number').min(),
    'max': df.select_dtypes('number').max(),
})
calidad

,dtype,nulos,valores_unicos,min,max
Administrative,int64,0,27,0.000,27.000
Administrative_Duration,float64,0,3335,0.000,"3,398.750"
BounceRates,float64,0,1872,0.000,0.200
Browser,int64,0,13,1.000,13.000
ExitRates,float64,0,4777,0.000,0.200
Informational,int64,0,17,0.000,24.000
Informational_Duration,float64,0,1258,0.000,"2,549.375"
Month,str,0,10,NaN,NaN
OperatingSystems,int64,0,8,1.000,8.000
PageValues,float64,0,2704,0.000,361.764


In [6]:
duplicadas = int(df.duplicated().sum())
negativos = int((df.select_dtypes('number') < 0).sum().sum())
tasa_con = df['Revenue'].mean()
tasa_sin = df.drop_duplicates()['Revenue'].mean()

print(f'Valores faltantes totales : {int(df.isna().sum().sum())}')
print(f'Valores numericos negativos: {negativos}')
print(f'Sesiones duplicadas exactas: {duplicadas} ({duplicadas / len(df):.2%})')
print(f'Tasa de compra con duplicados: {tasa_con:.2%} | sin duplicados: {tasa_sin:.2%}')

Valores faltantes totales : 0
Valores numericos negativos: 0
Sesiones duplicadas exactas: 125 (1.01%)
Tasa de compra con duplicados: 15.47% | sin duplicados: 15.63%


In [7]:
# Coherencia interna: si no se visitan paginas de un tipo, su duracion deberia ser 0
for pag, dur in [('Administrative', 'Administrative_Duration'), ('Informational', 'Informational_Duration'),
                 ('ProductRelated', 'ProductRelated_Duration')]:
    sin_paginas = df[pag] == 0
    print(f'{pag:<15} sin paginas: {sin_paginas.sum():>6,} sesiones | con duracion > 0 pese a 0 paginas: {(sin_paginas & (df[dur] > 0)).sum()}')

# Cardinalidad de las variables nominales
card = df[['Month', 'VisitorType', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']].nunique().rename('categorias').to_frame()
card

Administrative  sin paginas:  5,768 sesiones | con duracion > 0 pese a 0 paginas: 0
Informational   sin paginas:  9,699 sesiones | con duracion > 0 pese a 0 paginas: 0
ProductRelated  sin paginas:     38 sesiones | con duracion > 0 pese a 0 paginas: 0


,categorias
Month,10
VisitorType,3
OperatingSystems,8
Browser,13
Region,9
TrafficType,20


**Lectura de la auditoría y decisión asociada**

| Hallazgo | Magnitud | Decisión |
|---|---|---|
| Valores faltantes | **0** | No hace falta imputar |
| Valores negativos o inconsistentes | **0** (ninguna sesión tiene duración > 0 con 0 páginas) | Datos coherentes |
| Sesiones **duplicadas exactas** | **125** (1,0 %); todas son sesiones *sin compra* (la tasa sube de 15,47 % a 15,63 % al quitarlas) | **Eliminarlas antes de dividir** en train/test: si un duplicado cae en cada lado, el modelo "ve" la respuesta |
| Sesiones sin páginas administrativas / informativas | 46,8 % / 78,7 % | Esperable: no todos visitan esas secciones. Los muchos ceros piden transformar las variables (log) |
| Cardinalidad de las nominales | `TrafficType` 20, `Browser` 13, `Month` 10, `Region` 9, `OperatingSystems` 8 | **Ninguna es de cardinalidad realmente alta**: se puede usar one-hot (~73 columnas) |

### ⚠️ Advertencia metodológica sobre `PageValues`

Según la documentación del dataset, `PageValues` es *"el valor promedio de las páginas que el usuario visitó **antes de completar una transacción**"*. Es decir, se calcula en parte con información del **resultado** de la sesión: una sesión sin compra casi siempre tiene `PageValues = 0`.

Esto es un riesgo clásico de **fuga de información (data leakage)**: el modelo podría "predecir" la compra usando una variable que ya contiene la compra. No se descarta la variable sin evidencia; en cambio, el proyecto **medirá cuánto depende el modelo de ella** (notebook 03: modelo con y sin `PageValues`) y la **excluirá del agrupamiento** (notebook 04).

## 4. Plan de trabajo

| Notebook | Contenido |
|---|---|
| `01_problema_y_eda` | Problema, datos, calidad y EDA *(este)* |
| `02_preprocesamiento` | División train/test, pipeline sin fuga de información, selección de variables y PCA |
| `03_supervisado` | Regresión logística, Random Forest y Gradient Boosting; desbalance, umbral, importancia, ablación de `PageValues` |
| `04_no_supervisado` | K-Means, Gaussian Mixture y Aglomerativo; métricas, PCA/t-SNE, perfiles y relación con `Revenue` |
| `05_comparacion_y_conclusiones` | Respuestas a las 8 preguntas comparativas y recomendación |